In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from dotenv import load_dotenv



In [4]:
load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)


In [5]:
class SubgraphState(TypedDict):
    query: str
    answere: str

In [6]:
def process_query(state: SubgraphState):
    response = llm.invoke(state['query'])

    return{
        'answere': response.content
    }

In [8]:
subgraph_builder = StateGraph(SubgraphState)

subgraph_builder.add_node('process_query',process_query)

subgraph_builder.add_edge(START,'process_query')
subgraph_builder.add_edge('process_query',END)

subgraph = subgraph_builder.compile()

In [10]:
## parent node
class ParentState(TypedDict):
    question: str
    response: str

In [11]:
def call_subgraph(state: ParentState):
    subgraph_input ={
        'query': state['question']
    }


    subgraph_result = subgraph.invoke(subgraph_input)


    return{
        'response': subgraph_result['answere']
    }

In [12]:
parent_builder = StateGraph(ParentState)

parent_builder.add_node("subgraph", call_subgraph)

parent_builder.add_edge(START, "subgraph")
parent_builder.add_edge("subgraph", END)

parent = parent_builder.compile()


In [15]:
result = parent.invoke({
    'question': 'what is Ai'
})

print(result)

{'question': 'what is Ai', 'response': '**AI** stands for **Artificial Intelligence**—the field of computer science that builds systems capable of performing tasks that normally require human intelligence. These tasks include:\n\n| Domain | Typical AI Tasks |\n|--------|------------------|\n| **Perception** | Vision (image recognition), speech recognition, natural language understanding |\n| **Reasoning** | Planning, decision‑making, problem solving, logical inference |\n| **Learning** | Machine learning, deep learning, reinforcement learning |\n| **Interaction** | Chatbots, virtual assistants, recommendation engines |\n\n### Core Concepts\n\n| Concept | What it means |\n|---------|---------------|\n| **Algorithms** | Step‑by‑step procedures that process data |\n| **Models** | Mathematical representations (e.g., neural networks) that learn patterns |\n| **Training** | Feeding data to a model so it adjusts its internal parameters |\n| **Inference** | Using a trained model to make predic